The goal of this notebook is to repeat the three Gulf Stream lifetime analyses (`chl_over_lifetime.ipynb`, `pfts_over_lifetime.ipynb`, `pigments_over_lifetime.ipynb`) for the Kuroshio Extension. The experiment `kuroshio_20240305_20260531` covers 135 to 160 E and 29 to 44 N over the same dates, with the same detection, tracking, and coverage settings. The jet axis is traced the same way inside a 30 to 41 N band, and the open-ocean mask excludes the Sea of Japan instead of the Great Lakes.

What is known:
- North of the Kuroshio Extension is the cooler, nutrient-rich Kuroshio-Oyashio mixed water region. South of it is warmer, oligotrophic subtropical water.
- Warm-core anticyclones are shed north of the jet and cold-core cyclones south of it, as for the Gulf Stream.

Target eddies:
- Cyclones formed north of the Kuroshio axis and ended south, or formed within `NEAR_AXIS_KM` (150 km) of the axis and ended south. Anticyclones use the reversed rule.

Eddy requirements:
- Copernicus CHL and plankton groups: 50% CHL coverage of the speed-contour interior and at least 10 valid pixels per eddy-composite. A group keeps the same rule on its own pixels, because the group mask is smaller than the CHL mask.
- SDP pigments: 80% Rrs coverage of the speed-contour interior and at least 10 valid pixels per eddy-composite, from `collocate_pace` and `build_gold_table`.
- Within each age bin, all pixels within each eddy-composite are averaged, then the composites of each eddy are averaged, and the bootstrap resamples eddies.

Three sections, each with the interior mean per age bin and the mean in each age bin and 0.2 R ring out to 2 R, where R is the PET speed radius: Copernicus CHL, the nine Copernicus plankton groups, and the 13 SDP pigments. A ring needs 3 valid pixels, and a Copernicus ring also 50% coverage; a cell needs 3 eddies.

In [ ]:
from pathlib import Path
from typing import cast
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoLocator, MaxNLocator

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations
from eddy_tracking.preprocess.tracks import load_track_observations

EXPERIMENT = 'kuroshio_20240305_20260531'
N_AGE_BINS = 5
N_RADIAL_BINS = 10
MAX_RADIUS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
EXCLUDE_RECORD_EDGE_TRACKS = False
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
identity_columns = ['polarity', 'track_id']
groups = ['DIATO', 'DINO', 'GREEN', 'HAPTO', 'PROCHLO', 'PROKAR', 'MICRO', 'NANO', 'PICO']
group_labels = {
    'DIATO': 'Diatoms', 'DINO': 'Dinophytes', 'GREEN': 'Green algae', 'HAPTO': 'Haptophytes',
    'PROCHLO': 'Prochlorophytes', 'PROKAR': 'Prokaryotes',
    'MICRO': 'Microphytoplankton', 'NANO': 'Nanophytoplankton', 'PICO': 'Picophytoplankton',
}
pigments = ['Tchla', 'Zea', 'DV_chla', 'ButFuco', 'HexFuco', 'Allo', 'MV_chlb', 'Neo', 'Viola', 'Fuco', 'Chlc12', 'Chlc3', 'Perid']
pigment_labels = {
    'Tchla': 'Total chlorophyll-a', 'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
    'ButFuco': "19'-But-fucoxanthin", 'HexFuco': "19'-Hex-fucoxanthin", 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin', 'Fuco': 'Fucoxanthin',
    'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Perid': 'Peridinin',
}
panel_letters = 'abcdefghijklm'
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)

eddy_tracks = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
physical_start, physical_end = pd.to_datetime(cfg['base']['time']['eddy_date_range'])
eddy_tracks['at_record_edge'] = (
    (eddy_tracks['birth_date'] <= physical_start)
    | (eddy_tracks['death_date'] >= physical_end)
)
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['crossed_axis'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['near_axis_birth'] = (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM)
    & eddy_tracks['death_side'].eq(target_class.str[1])
)
eddy_tracks['is_target'] = eddy_tracks['crossed_axis'] | eddy_tracks['near_axis_birth']
plankton = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet').merge(
    eddy_tracks[identity_columns + ['at_record_edge', 'is_target']], on=identity_columns, how='left',
)
pigment_table = pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet')
pigment_table['polarity'] = cast(pd.Series, pigment_table['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
pigment_table = pigment_table.rename(columns={f'eddy_mean_{pigment}': pigment for pigment in pigments}).merge(
    eddy_tracks[identity_columns + ['at_record_edge', 'is_target']], on=identity_columns, how='left',
)
target_plankton = cast(pd.DataFrame, plankton.loc[plankton['is_target']]).copy()
target_composites = cast(pd.DataFrame, pigment_table.loc[pigment_table['is_target']]).copy()
if EXCLUDE_RECORD_EDGE_TRACKS:
    target_plankton = cast(pd.DataFrame, target_plankton.loc[~target_plankton['at_record_edge']]).copy()
    target_composites = cast(pd.DataFrame, target_composites.loc[~target_composites['at_record_edge']]).copy()
plankton_settings = cfg['collocate_plankton']
for group in groups:
    covered = (
        target_plankton[f'{group}_n_pixels'].ge(plankton_settings['min_pixels'])
        & target_plankton[f'{group}_n_pixels'].div(target_plankton['n_pixels']).ge(plankton_settings['min_coverage'])
    )
    target_plankton.loc[~covered, group] = np.nan

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})


def bin_by_age(table, fields):
    analysis = table.sort_values(identity_columns + ['date']).melt(
        id_vars=identity_columns + ['date', 'age_frac'], value_vars=fields,
        var_name='field', value_name='concentration',
    )
    analysis['age_bin'] = np.minimum(
        (analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1,
    )
    analysis['change'] = analysis['concentration'] - analysis.groupby(identity_columns + ['field'])['concentration'].transform('first')
    return analysis


def summarize_lifetime(analysis, fields):
    eddy_bins = cast(pd.DataFrame, analysis.groupby(identity_columns + ['field', 'age_bin']).agg(
        concentration=('concentration', 'mean'), change=('change', 'mean'),
    )).reset_index()
    rng = np.random.default_rng(RANDOM_SEED)
    summary_rows = []
    for polarity in polarity_names:
        polarity_bins = eddy_bins.loc[eddy_bins['polarity'].eq(polarity)]
        eddy_ids = sorted(polarity_bins['track_id'].unique())
        draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
        for field in fields:
            field_bins = polarity_bins.loc[polarity_bins['field'].eq(field)]
            for metric in ('concentration', 'change'):
                matrix = field_bins.pivot(index='track_id', columns='age_bin', values=metric).reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
                counts = np.isfinite(matrix).sum(axis=0)
                means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
                sampled = matrix[draws]  # (n_eddies, n_bins) -> (n_bootstrap, n_eddies, n_bins)
                sampled_counts = np.isfinite(sampled).sum(axis=1)
                sampled_means = np.divide(
                    np.nansum(sampled, axis=1), sampled_counts,
                    out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0,
                )
                for age_bin in range(N_AGE_BINS):
                    bootstrap_values = sampled_means[:, age_bin]
                    bootstrap_values = bootstrap_values[np.isfinite(bootstrap_values)]
                    low, high = (np.quantile(bootstrap_values, [0.025, 0.975]) if counts[age_bin] >= 3 else (np.nan, np.nan))
                    summary_rows.append({
                        'polarity': polarity, 'field': field, 'metric': metric, 'age_bin': age_bin,
                        'age_midpoint': bin_centers[age_bin], 'mean': means[age_bin],
                        'ci_low': low, 'ci_high': high, 'n_eddies': int(counts[age_bin]),
                    })
    return pd.DataFrame(summary_rows)


def draw_lifetime(summary, fields, labels, n_eddies, metric, ylabel, n_columns, figsize):
    n_rows = -(-len(fields) // n_columns)
    fig, axes = cast(tuple[Figure, np.ndarray], plt.subplots(n_rows, n_columns, figsize=figsize, sharex=True, layout='constrained'))
    spare = [cast(Axes, ax) for ax in axes.flat[len(fields):]]
    for ax in spare[1:]:
        ax.remove()
    for index, (letter, field) in enumerate(zip(panel_letters, fields)):
        ax = cast(Axes, axes.flat[index])
        if metric == 'change':
            ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
        for polarity in polarity_names:
            result = summary.loc[
                summary['polarity'].eq(polarity) & summary['field'].eq(field) & summary['metric'].eq(metric)
            ].sort_values('age_bin')
            color = polarity_colors[polarity]
            intervals = result.loc[result['ci_low'].notna()]
            ax.errorbar(
                intervals['age_midpoint'], intervals['mean'],
                yerr=[intervals['mean'] - intervals['ci_low'], intervals['ci_high'] - intervals['mean']],
                fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=2,
            )
            ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, linewidth=1.2, markersize=3.2, markeredgecolor='white', markeredgewidth=0.5, label=f'{target_labels[polarity]} (n = {n_eddies[polarity]})', zorder=3)
        ax.set_title(f'$\\bf{{({letter})}}$ {labels[field]}', loc='left', fontsize=8)
        ax.xaxis.set_ticks(np.linspace(0, 1, 6))
        ax.set_xlim(0, 1)
        ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
        ax.set_axisbelow(True)
        locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
        ax.yaxis.set_major_locator(locator)
        ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
        ax.set_ylim(ticks[0], ticks[-1])
        if index + n_columns >= len(fields):
            ax.tick_params(labelbottom=True)
            if not spare:
                ax.set_xlabel('Fraction of observed track')
    handles, handle_labels = cast(Axes, axes.flat[0]).get_legend_handles_labels()
    if spare:
        spare[0].axis('off')
        spare[0].legend(handles, handle_labels, loc='upper left', fontsize=8).set_in_layout(False)
        fig.supxlabel('Fraction of observed track', fontsize=8)
    else:
        fig.legend(handles, handle_labels, loc='outside lower center', ncol=2)
    fig.supylabel(ylabel, fontsize=8)
    plt.show()


def summarize_cells(rings, fields):
    radial_analysis = rings.melt(
        id_vars=identity_columns + ['date', 'age_frac', 'radial_bin'], value_vars=fields,
        var_name='field', value_name='concentration',
    )
    radial_analysis['age_bin'] = np.minimum(
        (radial_analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1,
    )
    eddy_cells = radial_analysis.groupby(identity_columns + ['field', 'age_bin', 'radial_bin'])['concentration'].mean().reset_index()
    cells = cast(pd.DataFrame, eddy_cells.groupby(['polarity', 'field', 'age_bin', 'radial_bin']).agg(
        concentration=('concentration', 'mean'), n_eddies=('concentration', 'count'),
    )).reset_index()
    cells.loc[cells['n_eddies'].lt(3), 'concentration'] = np.nan
    return cells


def draw_age_radius(cells, fields, labels, n_columns, figsize, margins, pair_wspace):
    n_rows = -(-len(fields) // n_columns)
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('#e6e6e6')
    fig = plt.figure(figsize=figsize)
    outer = fig.add_gridspec(n_rows, n_columns, **margins)
    for index, (letter, field) in enumerate(zip(panel_letters, fields)):
        field_cells = cells.loc[cells['field'].eq(field)]
        vmin, vmax = field_cells['concentration'].min(), field_cells['concentration'].max()
        axes = cast(np.ndarray, outer[index].subgridspec(1, 2, wspace=pair_wspace).subplots(sharey=True))
        for ax, polarity in zip(axes, polarity_names):
            ax = cast(Axes, ax)
            grid = field_cells.loc[field_cells['polarity'].eq(polarity)].pivot(index='radial_bin', columns='age_bin', values='concentration').reindex(index=range(N_RADIAL_BINS), columns=range(N_AGE_BINS)).to_numpy(dtype=float)
            ax.pcolormesh(bin_edges, radial_edges, np.ma.masked_invalid(grid), cmap=cmap, vmin=vmin, vmax=vmax, edgecolors='white', linewidth=0.5)
            ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
            ax.set_aspect('equal')
            ax.set_title(f'{polarity.capitalize()}s', fontsize=8, pad=2)
            ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'])
            ax.xaxis.set_ticks(bin_edges, minor=True)
            ax.yaxis.set_ticks([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1', '1.5', '2'])
            ax.yaxis.set_ticks(radial_edges, minor=True)
            ax.tick_params(labelsize=8, length=2)
            ax.tick_params(which='minor', length=1.2)
        colorbar = fig.colorbar(ScalarMappable(norm=Normalize(vmin, vmax), cmap=cmap), cax=cast(Axes, axes[1]).inset_axes((1.12, 0, 0.1, 1)))
        colorbar.ax.tick_params(labelsize=8, length=2)
        colorbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
        colorbar.outline.set_linewidth(0.5)
        cast(Axes, axes[0]).annotate(f'$\\bf{{({letter})}}$ {labels[field]}', (0, 1), xycoords='axes fraction', xytext=(0, 13), textcoords='offset points', fontsize=8, va='bottom', ha='left')
    fig.supxlabel('Fraction of observed track', fontsize=8)
    fig.supylabel('Distance from eddy center (speed radii)', fontsize=8)
    plt.show()


classes = ['NN', 'NS', 'SN', 'SS']
count_rows = []
matched_tracks = eddy_tracks.merge(
    cast(pd.Series, plankton.groupby(identity_columns).size()).rename('n_chl_dates').reset_index(),
    on=identity_columns,
)
fig, axes = plt.subplots(1, 2, figsize=(6.69, 2.7), sharey=True, layout='constrained')
for panel, polarity in enumerate(polarity_names):
    ax = cast(Axes, axes[panel])
    tracks = matched_tracks.loc[matched_tracks['polarity'] == polarity]
    total = tracks.groupby('movement').size().reindex(classes, fill_value=0)
    target = tracks.loc[tracks['is_target']].groupby('movement').size().reindex(classes, fill_value=0)
    positions = np.arange(len(classes))
    all_bars = ax.bar(positions - 0.19, total.to_numpy(), width=0.36, color='#c3c8cd', label='All tracks')
    target_bars = ax.bar(positions + 0.19, target.to_numpy(), width=0.36, color=polarity_colors[polarity], label='Target eddies')
    ax.bar_label(all_bars, padding=2, fontsize=8, color='#333333')
    ax.bar_label(target_bars, labels=[str(value) if value else '' for value in target.to_numpy()], padding=2, fontsize=8, color='#333333')
    ax.xaxis.set_ticks(positions)
    ax.xaxis.set_ticklabels([label[0] + ' → ' + label[1] for label in classes])
    ax.set_title(f'$\\bf{{({"ab"[panel]})}}$ {polarity.capitalize()}s: {len(tracks)} tracks, {int(target.sum())} target eddies', loc='left')
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', length=0)
    for label in classes:
        count_rows.append({'polarity': polarity, 'class': label, 'tracks': int(total[label]), 'target': int(target[label])})
axes[0].set_ylim(0, np.ceil(int(matched_tracks.groupby(['polarity', 'movement']).size().max()) * 1.12 / 50) * 50)
axes[0].set_ylabel('Tracks with CHL data')
axes[0].legend(loc='upper left', handlelength=1.2, handleheight=0.9)
fig.supxlabel('First side → last side of the daily Kuroshio axis', fontsize=8)
class_counts = pd.DataFrame(count_rows)
plt.show()
print('Tracks with CHL data and target eddies in each movement class, the counts behind the bars.')
display(class_counts)
target_rules = {
    'crossed the axis': eddy_tracks['crossed_axis'],
    f'born within {NEAR_AXIS_KM} km, no crossing': eddy_tracks['near_axis_birth'] & ~eddy_tracks['crossed_axis'],
    'target eddies': eddy_tracks['is_target'],
}
print(f'Target eddies by the rule that admits them: an axis crossing, or a birth within {NEAR_AXIS_KM} km of the axis without a crossing, and their sum.')
display(pd.DataFrame([
    {'polarity': polarity, 'rule': rule, 'tracks': int((members & eddy_tracks['polarity'].eq(polarity)).sum())}
    for polarity in polarity_names for rule, members in target_rules.items()
]))

In [ ]:
track_observations = load_track_observations(EXPERIMENT)
crossed = cast(pd.DataFrame, eddy_tracks.loc[eddy_tracks['crossed_axis']]).copy()
crossed['lifetime_days'] = (crossed['death_date'] - crossed['birth_date']).dt.days
representatives = crossed.sort_values('lifetime_days', ascending=False).drop_duplicates('polarity').set_index('polarity')

map_fig, map_ax = plt.subplots(figsize=(6.69, 4.3), subplot_kw={'projection': ccrs.PlateCarree()}, layout='constrained')
map_ax = cast(GeoAxes, map_ax)
map_ax.set_extent([*cfg['base']['region']['lon_range'], *cfg['base']['region']['lat_range']], crs=ccrs.PlateCarree())
map_ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=0)
map_ax.coastlines(resolution='50m', color='#666666', linewidth=0.5)
grid = map_ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', xlocs=range(135, 161, 5), ylocs=range(30, 45, 2))
grid.top_labels = False
grid.right_labels = False
grid.xlabel_style = {'size': 8}
grid.ylabel_style = {'size': 8}
handles = []
for polarity in polarity_names:
    track_id = int(representatives.loc[polarity, 'track_id'])
    track = cast(pd.DataFrame, track_observations.loc[
        track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(track_id)
    ]).sort_values('date')
    first, last = track.iloc[0], track.iloc[-1]
    color = polarity_colors[polarity]
    gap_groups = track['date'].diff().dt.days.gt(1).cumsum()
    for _, segment in track.groupby(gap_groups):
        map_ax.plot(segment['center_lon'], segment['center_lat'], color=color, linewidth=1.3, transform=ccrs.PlateCarree())
    map_ax.scatter(first['center_lon'], first['center_lat'], s=30, marker='o', facecolors='white', edgecolors=color, linewidths=1.2, zorder=5, transform=ccrs.PlateCarree())
    map_ax.scatter(last['center_lon'], last['center_lat'], s=36, marker='^', color=color, edgecolors='white', linewidths=0.6, zorder=5, transform=ccrs.PlateCarree())
    handles.append(Line2D([], [], color=color, linewidth=1.3, label=f'{polarity.capitalize()} {track_id}, {first["date"]:%Y-%m-%d} to {last["date"]:%Y-%m-%d}, {len(track)} detections'))
handles.append(Line2D([], [], linestyle='none', marker='o', markerfacecolor='white', markeredgecolor='#333333', markersize=5, label='First detection'))
handles.append(Line2D([], [], linestyle='none', marker='^', color='#333333', markersize=5.5, label='Last detection'))
map_ax.legend(handles=handles, loc='lower right', frameon=True, framealpha=0.95, edgecolor='none')
plt.show()

In [ ]:
chl_analysis = bin_by_age(target_plankton, ['CHL'])
chl_summary = summarize_lifetime(chl_analysis, ['CHL'])
target_summary = cast(pd.DataFrame, chl_analysis.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
    first_composite=('date', 'min'), last_composite=('date', 'max'),
))
record_edge_counts = target_plankton.loc[target_plankton['at_record_edge']].drop_duplicates(identity_columns).groupby('polarity').size()
target_summary['tracks_at_record_edge'] = record_edge_counts.reindex(target_summary.index, fill_value=0)

life_fig, life_axes = plt.subplots(1, 2, figsize=(6.69, 2.9), layout='constrained')
for panel, metric in enumerate(('concentration', 'change')):
    ax = cast(Axes, life_axes[panel])
    if metric == 'change':
        ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    for polarity in polarity_names:
        result = chl_summary.loc[
            chl_summary['polarity'].eq(polarity) & chl_summary['metric'].eq(metric)
        ].sort_values('age_bin')
        color = polarity_colors[polarity]
        intervals = result.loc[result['ci_low'].notna()]
        ax.errorbar(
            intervals['age_midpoint'], intervals['mean'],
            yerr=[intervals['mean'] - intervals['ci_low'], intervals['ci_high'] - intervals['mean']],
            fmt='none', ecolor=color, capsize=2, elinewidth=0.8, capthick=0.8, zorder=2,
        )
        ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, markersize=4, markeredgecolor='white', markeredgewidth=0.6, label=f'{target_labels[polarity]} (n = {target_summary.loc[polarity, "eddies"]})', zorder=3)
    ax.set_xlabel('Fraction of observed track')
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.set_xlim(0, 1)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ticks = cast(np.ndarray, AutoLocator().tick_values(*ax.get_ylim()))
    ax.set_ylim(ticks[0], ticks[-1])
life_axes[0].set_title('$\\bf{(a)}$ Interior CHL', loc='left')
life_axes[0].set_ylabel('CHL (mg m$^{-3}$)')
life_axes[1].set_title('$\\bf{(b)}$ Change from first observation', loc='left')
life_axes[1].set_ylabel('$\\Delta$CHL (mg m$^{-3}$)')
life_fig.legend(*life_axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=2)
plt.show()
print('Eddies, composites, composite date range, and tracks at the record edge per polarity.')
display(target_summary)
print('Mean interior CHL per bin with the 95% interval and the number of eddies behind it, the values of panel (a).')
display(chl_summary.loc[chl_summary['metric'].eq('concentration')].drop(columns=['field', 'metric']).round(4))

In [ ]:
print(f'Target eddies and composites in the Copernicus plankton table: the distinct eddies of each polarity, the eddy-composites that pass the CHL rule of {plankton_settings["min_coverage"]:.0%} coverage of the speed-contour interior and {plankton_settings["min_pixels"]} valid pixels, and the eddy-composites that keep each group after the same rule on the pixels of that group. {len(target_plankton.drop_duplicates(identity_columns))} of the {int(eddy_tracks["is_target"].sum())} target eddies have at least one composite.')
display(cast(pd.DataFrame, target_plankton.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
    **{group: (group, 'count') for group in groups},
)))
group_analysis = bin_by_age(target_plankton, groups)
group_summary = summarize_lifetime(group_analysis, groups)
group_eddies = group_analysis.groupby('polarity')['track_id'].nunique()
for metric, ylabel in (
    ('concentration', 'Group chlorophyll-a (mg m$^{-3}$)'),
    ('change', 'Change from first observation (mg m$^{-3}$)'),
):
    draw_lifetime(group_summary, groups, group_labels, group_eddies, metric, ylabel, 3, (6.69, 6.1))
print('Mean group concentration per bin, the values of the first figure, in mg/m³.')
display(group_summary.loc[group_summary['metric'].eq('concentration')].pivot(index='field', columns=['polarity', 'age_bin'], values='mean').reindex(groups).round(4))
print('Eddies per bin, the minimum over the nine groups.')
display(group_summary.loc[group_summary['metric'].eq('concentration')].groupby(['polarity', 'age_bin'])['n_eddies'].min().unstack('age_bin'))

In [ ]:
observations = []
for polarity in polarity_names:
    tracked = TrackEddiesObservations.load_file(str(DATA_DIR / f'silver/eddy_track/{polarity}/{polarity}_tracks.zarr'))
    observations.append(pd.DataFrame({
        'polarity': polarity, 'track_id': tracked.track.astype(int),
        'day': pd.to_datetime(tracked.time.astype(int), unit='D', origin=pd.Timestamp('1950-01-01')),
        'center_lon': (tracked.longitude + 180) % 360 - 180, 'center_lat': tracked.latitude,
        'radius_km': tracked.radius_s / 1000, 'virtual': tracked.virtual.astype(bool),
    }))
observations = pd.concat(observations, ignore_index=True)
observations = observations.loc[~observations['virtual']].merge(target_plankton[identity_columns].drop_duplicates(), on=identity_columns)
windows = []
for year in range(physical_start.year, physical_end.year + 1):
    start = pd.Timestamp(year, 1, 1)
    while start.year == year:
        end = min(start + pd.Timedelta(days=7), pd.Timestamp(year, 12, 31))
        if end >= physical_start and start <= physical_end:
            windows.append((start, end))
        start = end + pd.Timedelta(days=1)
plankton_fields = ['CHL'] + groups
fields = xr.open_mfdataset(sorted((DATA_DIR / 'bronze/plankton').glob('plankton_*.nc')), combine='by_coords')[plankton_fields]
lon = fields['longitude'].to_numpy()
lat = fields['latitude'].to_numpy()
plankton_rings = []
for start, end in windows:
    composites = target_plankton.loc[target_plankton['date'].between(start, end), identity_columns + ['date', 'age_frac']]
    if composites.empty:
        continue
    candidates = observations.loc[observations['day'].between(start, end)].merge(composites, on=identity_columns)
    candidates['offset'] = (candidates['day'] - candidates['date']).abs()
    composite = fields.sel(time=slice(start, end)).mean('time').load()
    for eddy in candidates.sort_values(['offset', 'day']).drop_duplicates(identity_columns).itertuples():
        half_width = MAX_RADIUS * eddy.radius_km / 111.32
        lon_index = np.flatnonzero(np.abs(lon - eddy.center_lon) <= half_width / np.cos(np.radians(eddy.center_lat)))
        lat_index = np.flatnonzero(np.abs(lat - eddy.center_lat) <= half_width)
        lon_grid, lat_grid = np.meshgrid(lon[lon_index], lat[lat_index])
        half_chord = (
            np.sin(np.radians(lat_grid - eddy.center_lat) / 2) ** 2
            + np.cos(np.radians(lat_grid)) * np.cos(np.radians(eddy.center_lat)) * np.sin(np.radians(lon_grid - eddy.center_lon) / 2) ** 2
        )
        distance_km = 2 * 6371 * np.arcsin(np.sqrt(half_chord))
        radial_bin = np.digitize(distance_km.ravel() / eddy.radius_km, radial_edges) - 1
        inside = radial_bin < N_RADIAL_BINS
        n_pixels = np.bincount(radial_bin[inside], minlength=N_RADIAL_BINS)
        ring = {
            'polarity': eddy.polarity, 'track_id': eddy.track_id, 'date': eddy.date, 'age_frac': eddy.age_frac,
            'radial_bin': np.arange(N_RADIAL_BINS), 'n_pixels': n_pixels,
        }
        box = composite.isel(longitude=lon_index, latitude=lat_index)
        for field in plankton_fields:
            values = box[field].to_numpy().ravel()
            valid = inside & np.isfinite(values)
            n_valid = np.bincount(radial_bin[valid], minlength=N_RADIAL_BINS)
            covered = (n_valid >= 3) & (n_valid >= plankton_settings['min_coverage'] * n_pixels)
            ring[field] = np.where(covered, np.bincount(radial_bin[valid], weights=values[valid], minlength=N_RADIAL_BINS) / np.maximum(n_valid, 1), np.nan)
            ring[f'{field}_n_pixels'] = n_valid
        plankton_rings.append(pd.DataFrame(ring))
plankton_rings = pd.concat(plankton_rings, ignore_index=True)
plankton_cells = summarize_cells(plankton_rings, plankton_fields)

draw_age_radius(plankton_cells, ['CHL'], {'CHL': 'Copernicus CHL'}, 1, (4.9, 4.0), dict(left=0.13, right=0.8, bottom=0.1, top=0.88), 0.12)
print('Mean CHL in each age bin (rows) and ring (columns, the index of the ring from the center), the values of the color map.')
display(plankton_cells.loc[plankton_cells['field'].eq('CHL')].pivot(index=['polarity', 'age_bin'], columns='radial_bin', values='concentration').round(3))
draw_age_radius(plankton_cells, groups, group_labels, 3, (6.69, 5.4), dict(left=0.075, right=0.92, bottom=0.075, top=0.935, wspace=0.6, hspace=0.6), 0.12)
print('Eddies per cell, the minimum over the nine groups, by ring (rows) and age bin (columns).')
display(plankton_cells.loc[plankton_cells['field'].isin(groups)].groupby(['radial_bin', 'polarity', 'age_bin'])['n_eddies'].min().unstack(['polarity', 'age_bin']))

In [ ]:
print(f'Target eddies and composites in the SDP pigment table: the distinct eddies of each polarity and the eddy-composites that pass the Rrs rule of {cfg["collocate_pace"]["min_coverage"]:.0%} coverage of the speed-contour interior and at least 10 valid pixels. Composite midpoints run from {target_composites["date"].min():%Y-%m-%d} to {target_composites["date"].max():%Y-%m-%d}, inside the PACE record. {len(target_composites.drop_duplicates(identity_columns))} of the {int(eddy_tracks["is_target"].sum())} target eddies have at least one composite.')
display(cast(pd.DataFrame, target_composites.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
)))
pigment_analysis = bin_by_age(target_composites, pigments)
pigment_summary = summarize_lifetime(pigment_analysis, pigments)
pigment_eddies = pigment_analysis.groupby('polarity')['track_id'].nunique()
for metric, ylabel in (
    ('concentration', 'Pigment concentration (mg m$^{-3}$)'),
    ('change', 'Change from first observation (mg m$^{-3}$)'),
):
    draw_lifetime(pigment_summary, pigments, pigment_labels, pigment_eddies, metric, ylabel, 4, (6.69, 7.6))
print('Mean pigment concentration per bin, the values of the first figure, in mg/m³.')
display(pigment_summary.loc[pigment_summary['metric'].eq('concentration')].pivot(index='field', columns=['polarity', 'age_bin'], values='mean').reindex(pigments).round(4))
print('Eddies per bin. The 13 pigments come from the same pixels, so the count is the same for each.')
display(pigment_summary.loc[pigment_summary['metric'].eq('concentration') & pigment_summary['field'].eq('Tchla')].pivot(index='polarity', columns='age_bin', values='n_eddies'))

In [ ]:
pixel_columns = {'T chla': 'Tchla', 'DV chla': 'DV_chla', 'MV chlb': 'MV_chlb', 'chl c1+c2': 'Chlc12', 'chl c3': 'Chlc3'}
pixels = pd.concat([
    pd.read_parquet(DATA_DIR / f'silver/pigments/{polarity}/eddy_{track_id}_pigments.parquet').assign(polarity=polarity)
    for polarity, track_id in target_composites[identity_columns].drop_duplicates().itertuples(index=False)
], ignore_index=True).rename(columns=pixel_columns)
pixels = pixels.merge(target_composites[identity_columns + ['date', 'age_frac']], on=identity_columns + ['date'])
half_chord = (
    np.sin(np.radians(pixels['pixel_lat'] - pixels['center_lat']) / 2) ** 2
    + np.cos(np.radians(pixels['pixel_lat'])) * np.cos(np.radians(pixels['center_lat'])) * np.sin(np.radians(pixels['pixel_lon'] - pixels['center_lon']) / 2) ** 2
)
pixels['radial_bin'] = np.digitize(2 * 6371 * np.arcsin(np.sqrt(half_chord)) / pixels['radius_km'], radial_edges) - 1
pigment_rings = cast(pd.DataFrame, pixels.loc[pixels['radial_bin'].lt(N_RADIAL_BINS)].groupby(identity_columns + ['date', 'age_frac', 'radial_bin']).agg(
    n_pixels=('Tchla', 'size'), **{pigment: (pigment, 'mean') for pigment in pigments},
)).reset_index()
pigment_rings.loc[pigment_rings['n_pixels'].lt(3), pigments] = np.nan
pigment_cells = summarize_cells(pigment_rings, pigments)

draw_age_radius(pigment_cells, pigments, pigment_labels, 3, (6.69, 8.9), dict(left=0.075, right=0.92, bottom=0.045, top=0.96, wspace=0.6, hspace=0.6), 0.12)
print('Eddies per cell by ring (rows) and age bin (columns). The 13 pigments come from the same pixels, so the count is the same for each.')
display(pigment_cells.loc[pigment_cells['field'].eq('Tchla')].pivot(index='radial_bin', columns=['polarity', 'age_bin'], values='n_eddies'))